# 02 — Attach political party

Read the v1 parquet built by `src/preprocessing.py`, match `page_name` and `bylines` against the 2022 AEC candidate list, add a `political_party` column (`PartyAb` or null), and write a new v2 parquet partitioned by party.

## 1. Spark session

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
import re

# Override the parquet output committer — the cluster default points at an EMR
# class whose JAR isn't on the classpath, which breaks .write.parquet otherwise.
spark = SparkSession.builder \
    .appName('FB_API_party_match') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

## 2. Paths

In [ ]:
IN_PATH        = '/user/s3348393/main/preprocessing/v1/parquet'
OUT_PATH       = '/user/s3348393/main/preprocessing/v2/parquet'
HOUSE_CSV      = '../data/2022_election_candidates.csv'
SENATE_CSV     = '../data/2022_senate_candidates.csv'
PARTIES_CSV    = '../data/aec_parties.csv'

## 3. Load v1 parquet

In [4]:
df = spark.read.parquet(IN_PATH)
print('Rows:', df.count())
df.printSchema()

Rows: 3128023
root
 |-- id: string (nullable = true)
 |-- page_id: string (nullable = true)
 |-- page_name: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- ad_creation_date: date (nullable = true)
 |-- ad_delivery_start_date: date (nullable = true)
 |-- ad_delivery_stop_date: date (nullable = true)
 |-- creative_bodies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_captions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_descs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_titles: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- currency: string (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publisher_platforms: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- demographic_distribution: array (nullable = true)
 |

## 4. Load 2022 candidates (House + Senate)

The House CSV (`DivisionNm, PartyAb, PartyNm, Surname, GivenNm`) and Senate CSV (`state, PartyAB, PartyNm, Surname, GivenNm` — `PartyAB` is pre-filled by [01_5_senate_party_codes.ipynb](01_5_senate_party_codes.ipynb)) share the columns we need for matching. Normalise the party-code column to `PartyAb` and concatenate into one DataFrame so the candidate-list loop in section 6 handles both as a single source.

In [ ]:
house  = pd.read_csv(HOUSE_CSV,  header=0)
senate = pd.read_csv(SENATE_CSV, header=0).rename(columns={'PartyAB': 'PartyAb'})

candidates = pd.concat(
    [house[['PartyAb', 'PartyNm', 'Surname', 'GivenNm']],
     senate[['PartyAb', 'PartyNm', 'Surname', 'GivenNm']]],
    ignore_index=True,
)

print('House candidates: ', len(house))
print('Senate candidates:', len(senate))
print('Combined:         ', len(candidates))
candidates.head()

## 5. Tokenise `page_name` and `bylines`

Use Spark MLlib `RegexTokenizer` (splits on `\W+`, lowercases) + `StopWordsRemover` with English defaults extended by political honorifics. Fields are tokenised independently — we won't concatenate them before matching, to avoid a given name in `page_name` crossing with a surname in `bylines` to spuriously match a candidate.

`coalesce(col, lit(''))` because `RegexTokenizer` errors on null inputs.

In [11]:
from pyspark.sql.functions import coalesce, col, lit
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

df = df.withColumn('page_name_safe', coalesce(col('page_name'), lit(''))) \
       .withColumn('bylines_safe',   coalesce(col('bylines'),   lit('')))

honorifics = ['mp', 'hon', 'dr', 'mr', 'mrs', 'ms', 'sen', 'senator', 'rt', 'authorised', 'by', 'for']
stopwords = StopWordsRemover.loadDefaultStopWords('english') + honorifics

for src, tok_col, term_col in [
    ('page_name_safe', 'page_name_tokens', 'page_name_terms'),
    ('bylines_safe',   'bylines_tokens',   'bylines_terms'),
]:
    df = RegexTokenizer(inputCol=src, outputCol=tok_col, pattern=r'\W+', toLowercase=True).transform(df)
    df = StopWordsRemover(inputCol=tok_col, outputCol=term_col, stopWords=stopwords).transform(df)

df.select('page_name', 'page_name_terms', 'bylines', 'bylines_terms').show(5, truncate=80)

+--------------+---------------+--------------+--------------+
|     page_name|page_name_terms|       bylines| bylines_terms|
+--------------+---------------+--------------+--------------+
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
+--------------+---------------+--------------+--------------+
only showing top 5 rows



## 6. Build candidate match index

For each candidate, the *required tokens* are the first given-name token plus all surname tokens — e.g. Adam ABDUL RAZAK → `{adam, abdul, razak}`. Token-set subset matching is order-insensitive, so 'Adam Abdul Razak' and 'Abdul Razak, Adam' both match.

Index by the first surname token to keep matching fast: at lookup time we only check candidates whose key token appears in the ad's token list, rather than scanning all ~1,500 candidates per ad.

In [ ]:
def split_tokens(s):
    if not isinstance(s, str):
        return []
    return [t for t in re.split(r'\W+', s.lower()) if t]

candidate_list = []
for _, row in candidates.iterrows():
    surname_toks = split_tokens(row['Surname'])
    if not surname_toks:
        continue
    given_toks = split_tokens(row['GivenNm'])
    required = frozenset(surname_toks + given_toks[:1])
    party = row['PartyAb']
    if not isinstance(party, str):
        continue
    candidate_list.append((required, party))

print('Candidates indexed:', len(candidate_list))

Candidates indexed: 1203


## 7. Classify ads

Each ad gets two columns:

- **`political_party`** — AEC party code (`ALP`, `LP`, …) when the ad is from a registered-party candidate or party central office. Null otherwise.
- **`match_type`** — how the classification was reached:
  - `candidate` — a 2022 candidate's name (House or Senate) appears in `page_name` or `bylines`.
  - `party_org` — a party-name signature from `aec_parties.csv` matches the byline (e.g. byline = "Australian Labor Party").
  - `government` — byline is a government department or agency (e.g. "Department of Social Services", "Australian Electoral Commission").
  - `null` — none of the above.

Priority order is candidate → party_org → government. When more than one candidate matches and they disagree on party, the candidate stage drops the row and lets it fall through to the byline-level checks. Filter on `match_type IS NULL AND bylines IS NOT NULL` to surface the residual high-volume political-looking bylines (NGOs, unions, advocacy orgs, third-party political advertisers) for further triage.

In [ ]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, StructType, StructField

# Party-name byline signatures: rows where byline_tokens is non-empty in aec_parties.csv,
# preserving CSV order so specific signatures (e.g. {liberal, national}) match before
# single-token signatures ({liberal}).
aec_parties = pd.read_csv(PARTIES_CSV, header=0)

party_byline_signatures = []
for _, row in aec_parties.iterrows():
    bt = row['byline_tokens']
    if not isinstance(bt, str) or not bt.strip():
        continue
    sig = frozenset(t.strip() for t in bt.split('+') if t.strip())
    party_byline_signatures.append((sig, row['party_ab']))

# Government byline signatures. Narrow enough to avoid private-sector collisions.
# {commonwealth} was dropped — too high a false-positive risk against Commonwealth Bank.
GOV_BYLINE_SIGNATURES = [
    frozenset({'department'}),                          # "Department of X" — covers most departments
    frozenset({'australian', 'government'}),            # "Australian Government" prefix
    frozenset({'australian', 'electoral', 'commission'}),
]

print('Party-org byline signatures: ', len(party_byline_signatures))
print('Government byline signatures:', len(GOV_BYLINE_SIGNATURES))

# Combined classifier — pandas UDF, returns a DataFrame with (political_party, match_type)
# per input row. Pandas UDFs use Arrow to batch data between JVM and Python workers,
# eliminating per-row serialisation overhead vs a plain @udf — typically 3-10× faster.
result_schema = StructType([
    StructField('political_party', StringType(), True),
    StructField('match_type',      StringType(), True),
])

def _classify_one(page_tokens, bylines_tokens):
    # pandas_udf hands array columns to Python as numpy arrays, not lists.
    # `array or []` would call bool(array) — ambiguous for multi-element arrays.
    # Use explicit None checks to convert to set safely.
    page_set    = set(page_tokens)    if page_tokens    is not None else set()
    bylines_set = set(bylines_tokens) if bylines_tokens is not None else set()

    # 1. Candidate name match (House + Senate)
    parties = set()
    for required, party in candidate_list:
        if required.issubset(page_set) or required.issubset(bylines_set):
            parties.add(party)
    if len(parties) == 1:
        return (next(iter(parties)), 'candidate')
    # Ambiguous (>1 party) or no match — fall through.

    # 2. Party-name byline signature
    for sig, party in party_byline_signatures:
        if sig.issubset(bylines_set):
            return (party, 'party_org')

    # 3. Government byline signature
    for sig in GOV_BYLINE_SIGNATURES:
        if sig.issubset(bylines_set):
            return (None, 'government')

    return (None, None)


@pandas_udf(result_schema)
def classify_ad(page_terms: pd.Series, bylines_terms: pd.Series) -> pd.DataFrame:
    results = [_classify_one(p, b) for p, b in zip(page_terms, bylines_terms)]
    return pd.DataFrame(results, columns=['political_party', 'match_type'])


df = df.withColumn('_class', classify_ad('page_name_terms', 'bylines_terms'))
df = df.withColumn('political_party', col('_class.political_party')) \
       .withColumn('match_type',      col('_class.match_type')) \
       .drop('_class')

## 8. Sanity checks

In [16]:
df.groupBy('political_party').count().orderBy(col('count').desc()).show(50, truncate=False)

+---------------+-------+
|political_party|count  |
+---------------+-------+
|NULL           |2647852|
|ALP            |204812 |
|LP             |130105 |
|IND            |60868  |
|LNP            |15573  |
|GRN            |14646  |
|NP             |14123  |
|ON             |9103   |
|GVIC           |8444   |
|UAPP           |7010   |
|LDP            |4837   |
|JLN            |3069   |
|TNL            |2713   |
|CLP            |895    |
|XEN            |891    |
|CYA            |414    |
|AUVA           |370    |
|SOPA           |365    |
|VNS            |360    |
|KAP            |321    |
|IMO            |243    |
|TLOC           |220    |
|SAL            |211    |
|NaN            |178    |
|AJP            |172    |
|DPDA           |167    |
|GAP            |45     |
|REAS           |9      |
|SPP            |7      |
+---------------+-------+



In [ ]:
# breakdown by match_type
df.groupBy('match_type').count().orderBy(col('count').desc()).show(truncate=False)

### 8a. Visualisations

The same diagnostics, plotted for the report. Each chart aggregates in Spark on the cached `df`, then materialises a small result to pandas for matplotlib rendering — no full corpus collection.

Four views:

1. **Classification breakdown** — donut showing the candidate / party_org / government / unclassified split.
2. **Top parties** — horizontal bar of classified ad counts (excludes nulls).
3. **Classification source per party** — for the top 8 parties, the candidate-name match vs party-org byline fallback contribution.
4. **Weekly spend by classification** — temporal preview. The election spike in `candidate` + `party_org` against the steady-state `government` / `unclassified` baselines is the headline narrative for the temporal analysis in notebook 05.

In [ ]:
import matplotlib.pyplot as plt

# Consistent palette for the four classification categories — reused across
# every chart in this section so they read as one figure family.
MT_COLORS = {
    'candidate':    '#1f77b4',
    'party_org':    '#ff7f0e',
    'government':   '#2ca02c',
    'unclassified': '#7f7f7f',
}

# Headline classification breakdown — what fraction of ads landed in each bucket.
mt = df.groupBy('match_type').count().toPandas()
mt['match_type'] = mt['match_type'].fillna('unclassified')
mt = mt.set_index('match_type').reindex(MT_COLORS.keys()).dropna()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    mt['count'],
    labels=[f'{k}\n{int(v):,}' for k, v in mt['count'].items()],
    colors=[MT_COLORS[k] for k in mt.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.4),
)
ax.set_title('Ad classification breakdown')
plt.tight_layout()
plt.show()

In [ ]:
# Top 12 parties by classified ad count.
party_counts = (df.filter(col('political_party').isNotNull())
                  .groupBy('political_party').count()
                  .orderBy(col('count').desc()).limit(12)
                  .toPandas())

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(party_counts['political_party'][::-1], party_counts['count'][::-1], color='#1f77b4')
ax.set_xlabel('Ad count')
ax.set_title('Top 12 parties by classified ad count')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Classification source per party — for the top 8 parties, how much comes via
# candidate-name match vs party-org byline fallback. Government rows have null
# political_party so they don't appear here.
top_parties = [r['political_party'] for r in
               df.filter(col('political_party').isNotNull())
                 .groupBy('political_party').count()
                 .orderBy(col('count').desc()).limit(8).collect()]

compo = (df.filter(col('political_party').isin(top_parties))
           .groupBy('political_party', 'match_type').count()
           .toPandas())

pivot = compo.pivot(index='political_party', columns='match_type', values='count').fillna(0)
pivot = pivot.reindex(top_parties)
order = [c for c in ['candidate', 'party_org'] if c in pivot.columns]
pivot = pivot[order]

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot.bar(stacked=True, ax=ax, color=[MT_COLORS[c] for c in pivot.columns])
ax.set_xlabel('Party')
ax.set_ylabel('Ad count')
ax.set_title('Classification source per party (top 8)')
ax.legend(title='match_type', loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Weekly spend by match_type — temporal preview of the classifier's separation power.
# Aggregate in Spark on the cached df, then materialise to pandas for matplotlib.
from pyspark.sql.functions import date_trunc, sum as spark_sum
import pandas as pd

weekly = (df.filter(col('ad_seq_no') == 1)
            .filter(col('spend_mid').isNotNull())
            .withColumn('week', date_trunc('week', 'ad_creation_date'))
            .groupBy('week', 'match_type')
            .agg(spark_sum('spend_mid').alias('spend'))
            .orderBy('week')
            .toPandas())

weekly['match_type'] = weekly['match_type'].fillna('unclassified')
pivot = weekly.pivot(index='week', columns='match_type', values='spend').fillna(0)
order = [c for c in ['candidate', 'party_org', 'government', 'unclassified'] if c in pivot.columns]
pivot = pivot[order]

fig, ax = plt.subplots(figsize=(13, 5))
pivot.plot.area(ax=ax, color=[MT_COLORS[c] for c in pivot.columns], alpha=0.75)
ax.axvline(pd.Timestamp('2022-05-21'), linestyle='--', color='red', linewidth=1.5)
ax.text(pd.Timestamp('2022-05-22'), ax.get_ylim()[1] * 0.95, ' Election day',
        color='red', va='top')
ax.set_title('Weekly spend (latest snapshot) by classification — ad-creation week')
ax.set_ylabel('Spend ($)')
ax.set_xlabel('Week')
ax.legend(title='match_type', loc='upper right')
plt.tight_layout()
plt.show()

In [20]:
# spot-check: a few matched rows per top party
from pyspark.sql.functions import desc

top_parties = [r['political_party'] for r in
               df.filter(col('political_party').isNotNull())
                 .groupBy('political_party').count()
                 .orderBy(desc('count')).limit(5).collect()]

for p in top_parties:
    print(f'\n=== {p} ===')
    df.filter(col('political_party') == p).select('page_name', 'bylines', 'political_party').distinct().show(10, truncate=60)


=== ALP ===


+--------------------------------------+--------------------------------+---------------+
|                             page_name|                         bylines|political_party|
+--------------------------------------+--------------------------------+---------------+
|                     Kristina Keneally|               Kristina Keneally|            ALP|
|                       Alicia Payne MP|                 Alicia Payne MP|            ALP|
|      Tabatha Young - Labor for Bonner|Tabatha Young - labor for Bonner|            ALP|
|                            Mary Doyle|       Mary Judith Jacinta Doyle|            ALP|
|                        Anika Wells MP|                  Anika Wells MP|            ALP|
|                     Richard Marles MP|               Richard Marles MP|            ALP|
|Bronwen English - WA Labor for Forrest|                        WA Labor|            ALP|
|Andrew Charlton - Labor for Parramatta|                 Andrew Charlton|            ALP|
|        A

+-------------------------------------+-----------------------------------------------+---------------+
|                            page_name|                                        bylines|political_party|
+-------------------------------------+-----------------------------------------------+---------------+
|                         Zoe McKenzie|Liberal Party of Australia (Victorian Division)|             LP|
|      Shawn Lock - Liberal for Spence|                      South Australian Liberals|             LP|
|                            Ken Wyatt|                                      Ken Wyatt|             LP|
|                       Jason Falinski|                                 Jason Falinski|             LP|
|                      Angus Taylor MP|                                Angus Taylor MP|             LP|
|                        Richard Welch|Liberal Party of Australia (Victorian Division)|             LP|
|                     Celia Hammond MP|                         

+-----------------------------------------------------+--------------------------------------+---------------+
|                                            page_name|                               bylines|political_party|
+-----------------------------------------------------+--------------------------------------+---------------+
|                                        Zali Steggall|                         Zali Steggall|            IND|
|                                         Jack Dempsey|              Jack Dempsey for Hinkler|            IND|
|               North Sydney's Kylea Tink for Canberra|                            Kylea Tink|            IND|
|                 Liz Habermann - Independent for Grey|                         Liz Habermann|            IND|
|Matt Sharpham - Independent Candidate for New England|                Matthew Peter Sharpham|            IND|
|                            Rob Priestly for Nicholls|            Rob Priestly for Nicholls |            IND|
|

+-----------------------------------------------+------------------------------------+---------------+
|                                      page_name|                             bylines|political_party|
+-----------------------------------------------+------------------------------------+---------------+
|                                Michelle Landry|                     Michelle Landry|            LNP|
|                                  Angie Bell MP|                       Angie Bell MP|            LNP|
|            Colin Boyce MP - Member for Callide| Colin Boyce MP - Member for Callide|            LNP|
|                                  Ross Vasta MP|                       Ross Vasta MP|            LNP|
| Andrew Wallace - LNP Federal Member for Fisher|Liberal National Party of Queensland|            LNP|
|                               Stuart Robert MP|                    Stuart Robert MP|            LNP|
|                   Colin Boyce  - LNP for Flynn|                        

+--------------------------------------------------+--------------------------+---------------+
|                                         page_name|                   bylines|political_party|
+--------------------------------------------------+--------------------------+---------------+
|        Natasa Sojic - Greens Candidate for Fenner|                ACT Greens|            GRN|
|Kristyn Glanville - Greens candidate for Curl Curl|    The Greens NSW - Manly|            GRN|
|          Elizabeth Watson-Brown - Greens for Ryan|     The Australian Greens|            GRN|
|             Rachael Jacobs - Greens for Grayndler|            The Greens NSW|            GRN|
|                               Eli Davern - Greens|            The Greens NSW|            GRN|
|                Taylor Vandijk - Greens for Barton|            The Greens NSW|            GRN|
|                Asha Worsteling - Greens for Oxley|         Queensland Greens|            GRN|
|                 Mandy Nolan - Greens f

In [23]:
# eyeball false positives: highest-spend matched rows
df.filter(col('political_party').isNotNull() & col('spend_mid').isNotNull() & (col('ad_seq_no') == 1)) \
  .orderBy(col('spend_mid').desc()) \
  .select('page_name', 'bylines', 'political_party', 'spend_mid') \
  .show(20, truncate=60)

+-------------------------------------------------+---------------------------------------------------+---------------+---------+
|                                        page_name|                                            bylines|political_party|spend_mid|
+-------------------------------------------------+---------------------------------------------------+---------------+---------+
|                                  Josh Frydenberg|                                    Josh Frydenberg|             LP|  12499.5|
|                                    Alan Tudge MP|                                      Alan Tudge MP|             LP|   9499.5|
|  Simon Kennedy - Liberal Candidate for Bennelong|Liberal Party of Australia New South Wales Division|             LP|   9499.5|
|                                  Josh Frydenberg|                                    Josh Frydenberg|             LP|   9499.5|
|                                  Josh Frydenberg|                                    Jos

In [ ]:
# Residual third-party political-looking bylines:
# match_type IS NULL excludes candidate, party_org, AND government matches —
# what's left is non-party non-government political-adjacent advertising
# (NGOs, unions, advocacy orgs, industry bodies, individual political advertisers).

df.filter(col('match_type').isNull() & col('bylines').isNotNull()) \
  .groupBy('bylines').count() \
  .orderBy(desc('count')) \
  .show(30, truncate=80)

### Party-org spot check

After the classifier, the major party-org bylines should be tagged via the `party_org` `match_type`. The query below filters for `political_party IS NULL` *and* the specific byline — both should return **empty** result sets if the fallback caught them. Any rows that come back are central-office ads the signatures missed, worth triaging into `aec_parties.csv`.

In [27]:
from pyspark.sql.functions import element_at, substring

party_bylines = ['Australian Labor Party', 'Liberal Party of Australia']

for b in party_bylines:
    print(f'\n=== {b} ===')
    df.filter(col('political_party').isNull() & (col('bylines') == b)) \
      .select(
          'page_name',
          'bylines',
          substring(element_at('creative_bodies', 1), 1, 120).alias('body'),
      ) \
      .distinct() \
      .show(10, truncate=120)


=== Australian Labor Party ===


+----------------------+----------------------+------------------------------------------------------------------------------------------------------------------------+
|             page_name|               bylines|                                                                                                                    body|
+----------------------+----------------------+------------------------------------------------------------------------------------------------------------------------+
|Australian Labor Party|Australian Labor Party|Only Anthony Albanese and Labor have a plan for a better future for ALL Australians. \n\nOn May 21, you can vote for ...|
|Australian Labor Party|Australian Labor Party|                   🚨 EXCLUSIVE 🚨 \n\nScott Morrison’s LEAKED plan to tackle the rising cost of living for Australians.|
|             ACT Labor|Australian Labor Party|   ACT needs a strong woman in the Senate. Canberra need Katy’s progressive voice. Vote 1 Labor to elect Katy 

+--------------------------+--------------------------+------------------------------------------------------------------------------------------------------------------------+
|                 page_name|                   bylines|                                                                                                                    body|
+--------------------------+--------------------------+------------------------------------------------------------------------------------------------------------------------+
|Liberal Party of Australia|Liberal Party of Australia|                                                                                    Our plan is working. #StrongerFuture|
|Liberal Party of Australia|Liberal Party of Australia|We want to further help Australians get past the biggest hurdle on their path to home ownership - saving for a deposit -|
|Liberal Party of Australia|Liberal Party of Australia|                                                            

## 9. Write v2 parquet partitioned by party

Drop the intermediate token/term columns before writing — they're large arrays and easy to rebuild. Rows with `political_party = null` land in `political_party=__HIVE_DEFAULT_PARTITION__/` (expected; that's the bulk of the corpus).

In [ ]:
intermediate = ['page_name_safe', 'bylines_safe',
                'page_name_tokens', 'bylines_tokens',
                'page_name_terms', 'bylines_terms']

out = df.drop(*intermediate)

# Defensive: re-override the parquet committer at write time. SparkSession.builder
# in cell 2 is a no-op if a session is already running, so an old kernel-state may
# still carry the cluster's broken EMR default.
spark.conf.set('spark.sql.parquet.output.committer.class',
               'org.apache.parquet.hadoop.ParquetOutputCommitter')

out.write \
   .partitionBy('political_party') \
   .option('mapreduce.fileoutputcommitter.algorithm.version', '2') \
   .parquet(OUT_PATH, mode='overwrite')
print('Wrote:', OUT_PATH)

In [ ]:
# round-trip check
rt = spark.read.parquet(OUT_PATH)
print('Roundtrip rows:', rt.count())
rt.printSchema()